# Stackelberg-Duopol

We will analyze a two-firm market as the set up is given by the Stackelberg-Duopol (described in Stackelberg, H. (1934): Marktform und Gleichgewicht). We assume that both firms produce the same homogenous good. One of the firms produces first (is the Stackelberg leader). Then, the second firm produces after observing how much the leading firm produced. Since the leader firm knows that it is observed by the following firm, it incorporates this knowlegde into its decision process of how much of the good should be produced. By using subgame perfect Nash equilibrium we encounter how much of the good both firms produce in a Stackelberg-Duopol.

Imports and set magics:

In [273]:
import numpy as np
from scipy import optimize # for numerical solutions
import sympy as sm
import ipywidgets as widgets # for interactive plots/buttons

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

from darja_modelproject import stackelbergduopolClass
model = stackelbergduopolClass()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In general, we define L as the leader and F as the follower when having two firms: 
$$ L \in \{1,2\} \quad \text{and} \quad F \in \{1,2\} / \{L\}.$$ 


In the following, we assume that the first firm is the leader and the second firm is the follower. 

L chooses an amount of $x_{L}$, F chooses $x_{F}$ to produce. We assume that the two firms have different cost functions $C_{L} \neq  C_{F}$.

The Stackelberg Duopol is solved by backward induction. So, we first need to derive the best response function of the follower. 

F maximizes its protfit by: 
$$ \max_{x_{F} \geq 0} \quad P(x_{L}+ x_{F}) \cdot x_{F} - C_{F}(x_{F}) $$ 

To get the best response function of F we need to derive the profit function, $\partial/\partial x_F $, and solve it such that $x_F$ can be written as a function of $x_L$.

$$ \partial/\partial x_F = 0 \Leftrightarrow x_{F}^*  =  ... $$

L anticipates the optimal solution of F, $x_{F}^*$, and maximizes its profit by: 

$$ \max_{x_{L} \geq 0} \quad P(x_{L} +x_{F}^*) \cdot x_{L} - C_{L}(x_L)$$ 

Finally, L gets the optimal solution of $x_{L}^*$ when deriving its profit function and solving the FOC, $ \partial/\partial x_L = 0 $.

We define the inverse demand function as $\text{P}(x)$:

with $ x = x_L + x_F$:
$$

\text{P}(x) =
\begin{cases} 
 a - b\cdot(x_L + x_F), \quad \text{if} \quad (x_L + x_F) < a/b, \\
0,  \quad \text{if} \quad (x_L + x_F) \geq  a/b.
\end{cases} 
$$

We assume linear cost functions for both firms, respectively. These don't need to be exactly the same cost functions $(p_L \neq p_F)$.
$$
C_L(x_L) = p_L \cdot x_L
$$ 
$$
C_F(x_F) = p_F \cdot x_F. 
$$


## Analytical solution

There is an analytical solution for the Stackelberg Duopol. Therefore, we will first solve the model analytically by using sympy. 

We start by definining the parameters and variables in sympy: 

In [274]:
## solution using sympy 
x_1 = sm.symbols("x_1")
x_2 = sm.symbols("x_2")
a = sm.symbols("a")
b = sm.symbols("b")
p_1 = sm.symbols("p_1")
p_2 = sm.symbols("p_2")


Then we define the inverse demand function:

In [275]:
inverse_demand =  a-b*(x_1 + x_2)
inverse_demand

a - b*(x_1 + x_2)

Next, we define the objective function for the leading firm: 

In [276]:
objective_1 = inverse_demand * x_1 - p_1*x_1
objective_1

-p_1*x_1 + x_1*(a - b*(x_1 + x_2))

Now, we define the objective function for the following firm:

In [277]:
objective_2 = inverse_demand * x_2 - p_2*x_2
objective_2

-p_2*x_2 + x_2*(a - b*(x_1 + x_2))

The next step is to derive the objective function for the following firm regarding the amount $x_F$ the firm produces:

In [278]:
foc = sm.diff(objective_2, x_2)
foc

a - b*x_2 - b*(x_1 + x_2) - p_2

Afterwards, we solve the derivative such that $x_F$ only depends on the amount that the leader firm consumes, $x_L$:

In [279]:
sol = sm.solve(sm.Eq(foc,0), x_2)
sol ## best answer function of firm 2 

[(a - b*x_1 - p_2)/(2*b)]

We substitute x_F in the objective function of the leading firm by the previous solution, $x_F^*$, such that the objective function of the leading firm only depends on $x_L$:

In [280]:
sub_objective_1= objective_1.subs(x_2, sol[0])
sub_objective_1

-p_1*x_1 + x_1*(a - b*(x_1 + (a - b*x_1 - p_2)/(2*b)))

Finally, we can solve the FOC for the leading firm and get the optimal amount, $x_L^*$, it produces: 

In [281]:
foc_1 = sm.diff(sub_objective_1, x_1)
foc_1
sol_1 = sm.solve(sm.Eq(foc_1,0), x_1)
sol_1

[(a - 2*p_1 + p_2)/(2*b)]

Now, we insert some values for the parameters a,b, $p_L$, $p_F$:
$$ a = 5 $$
$$ b = 1/4 $$ 
$$ p_L = 2 $$
$$ p_F = 1. $$

In [282]:
sol_1[0].subs(a, 5).subs(b, 1/4).subs(p_1,2).subs(p_2,1)

4.00000000000000

So the leading firm produces $x_L^*$ = 4 units of the good. 

Having $x_L^*$ = 4 we can directly find out how much the following firm will produce when it observes how much the leader produces: 

We just substitute the optimal amount of the leading firm in the best response function of the following firm: 

In [283]:
sol_2 = sol[0].subs(x_1, 4).subs(a, 5).subs(b, 1/4).subs(p_1,2).subs(p_2,1)
sol_2 

6.00000000000000

So this is the amount both firms produce when the first firm is the leading firm: 
$$ x_L^*= 4 $$
$$ x_F^*= 6 $$

## Numerical solution

We still use the same price function and cost functions. 
This is a rather brute force approach but we get the same solution as trying analytically.

We use different techniques to solve the Stacklberg Duopol numerically. 
First, use a solver from scipy. Then we use a self-defined solver (similar to the one defined in the lecture). 
At the end we try to find the root of the first derivative of the leading firm's profit function which gives the optimal solution.

Here we use scipy and a solver to find the optimal amount:

In [284]:
def constraint_x(x):
    return x  # This ensures that x is non-negative


constraints ={'type': 'ineq', 'fun': constraint_x} ## adding constraint for x


## bounds for solutions
bounds = [(0,None)] ## non negative amount x

# c. call solver, use SLSQP
initial_guess = 6
## slsqp as a method can deal with bounds und constrains
sol_case2 = optimize.minimize(
model.neg_objective_1, initial_guess, bounds = bounds ,constraints= constraints,
method='SLSQP')


In [285]:
sol_case2

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -1.9999999999999991
       x: [ 4.000e+00]
     nit: 3
     jac: [-5.960e-08]
    nfev: 6
    njev: 3

In [286]:
sol_case2.x[0]

3.9999999999999996

And the solution for the second firm as the follower:

In [287]:
model.best_func2(sol_case2.x[0]) ## solution for firm 2

6.0

### Another approach to the find numerically the optimal amount:

Here, we use a self-defined solver, similar to the one in the lecture:

In [288]:
model.minimize_solver(20) ## own defined solver  

(4.0001073741824, 13, 79, 0)

It gives us the same solution as with using a solver from scipy. 

OR:

We only solve the FOC by finding the root of the first derivative of the leader's profit function.

In [289]:
## finding the root of the first derivative gives us the solution for the leading firm! 
optimize.root_scalar(model.derivative_1,x0=-5.0,method='newton')

      converged: True
           flag: converged
 function_calls: 3
     iterations: 1
           root: 4.0

# Further analysis

We have interactive plots to visualize the difference in the optimal solution when changing the parameters of the model.

In the first plot, we only consider the optimal solutions for follower and leader.

In [290]:
widgets.interact(model.interactive_figure_sol, 
                 p1 =widgets.FloatSlider(description=r"p1", min=0, max=3, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p2", min=0, max=3, step=0.05, value=1), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p1', max=3.0, step=0.05), FloatSlider(value=1.0, des…

In the second plot, we also consiider the profit of each firm depending on the optimal solution.

In [291]:
## plotting profit of leader and follower
widgets.interact(model.interactive_figure_profit, 
                 p1 =widgets.FloatSlider(description=r"p1", min=0, max=3, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p2", min=0, max=3, step=0.05, value=1), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p1', max=3.0, step=0.05), FloatSlider(value=1.0, des…

# Extension


We will extend the market by adding another firm such that we have an oligopol. We still assume that the first firm is the leader and produces first. After that, firm 2 and firm 3 produce as followers. Now, this second stage is a Cournot competition between firm 2 and firm 3 since the following firms decide simultaneously how much to produce. Therefore, we first have to look at the FOC of the two following firms, solve the equation system such that the amount both produce can be written as function of how much the leading firm produces ($x_L$) and insert this in the profit function for the leading firm. Now the profit function of the leading firm only depends on $x_L$ and by the usual FOC we find again $x_L^*$ and afterwards $x_{F1}^*$ and $x_{F2}^*$.

F maximizes its protfit by: 
$$ \max_{x_{F} \geq 0} \quad P(x_{L}+ x_{F1} + x_{F2}) \cdot x_{F} - C_{F}(x_{F}) $$ 

To get the best response function of F1 and F2 we need to derive $\partial/\partial x_{F1}$ and  $\partial/\partial x_{F2}$ solve the two equations simultaneously such that $x_{F1}$ and $x_{F2}$ are written only as a function of $x_L$.

$$ \partial/\partial x_{F1} = 0 \Leftrightarrow x_{F1}^*  =  ... $$
$$ \partial/\partial x_{F2} = 0 \Leftrightarrow x_{F2}^*  =  ... $$

L anticipates the optimal solution of the two followers, $x_{F1}^*$ anf $x_{F2}^*$, and maximizes its profit by: 

$$ \max_{x_{L} \geq 0} \quad P(x_{L} +x_{F1}^* + x_{F2}^*) \cdot x_{L} - C_{L}(x_L)$$ 

Finally, L gets the optimal solution of $x_{L}^*$ when solving the FOC of $ \partial/\partial x_L = 0 $.

We still assume the same inverse demand function for all three firms as in the Stackelberg-Duopol.

With $$x = x_L + x_{F1} + x_{F2}$$:
$$
\text{P}(x) =
\begin{cases} 
 a - b\cdot(x_L + x_{F1} + x_{F2}), \quad \text{if} \quad (x_L + x_{F1} + x_{F2}) < a/b, \\
0,  \quad \text{if} \quad (x_L + x_{F1} + x_{F2}) \geq  a/b.
\end{cases} 
$$

### Analytical solution of the extension: 

First, we use sympy to solve the model analytically: 

In [292]:
## solution using sympy 
x_1 = sm.symbols("x_1")
x_2 = sm.symbols("x_2")
x_3 = sm.symbols("x_3")
a = sm.symbols("a")
b = sm.symbols("b")
p_1 = sm.symbols("p_1")
p_2 = sm.symbols("p_2")
p_3 = sm.symbols("p_3")

Since we have three firms we define inverse demand and objective functions again:

In [293]:
inverse_demand_extend =  a-b*(x_1 + x_2 + x_3)
inverse_demand_extend

a - b*(x_1 + x_2 + x_3)

Extended objective function for the first firm (leader):

In [294]:
objective_1_extend = inverse_demand_extend * x_1 - p_1*x_1
objective_1_extend

-p_1*x_1 + x_1*(a - b*(x_1 + x_2 + x_3))

Extended objective function for the second firm (follower): 

In [295]:
objective_2_extend = inverse_demand_extend * x_2 - p_2*x_2
objective_2_extend

-p_2*x_2 + x_2*(a - b*(x_1 + x_2 + x_3))

Extended objective function for the third firm (follower):

In [296]:
objective_3_extend = inverse_demand_extend * x_3 - p_3*x_3
objective_3_extend

-p_3*x_3 + x_3*(a - b*(x_1 + x_2 + x_3))

Now we need the FOC of the second and third firm:

In [297]:
foc2_extend = sm.diff(objective_2_extend, x_2)
foc2_extend
sol2_extend = sm.solve(sm.Eq(foc2_extend,0), x_2)
sol2_extend

[(a - b*(x_1 + x_3) - p_2)/(2*b)]

In [298]:
foc3_extend = sm.diff(objective_3_extend, x_3)
foc3_extend

a - b*x_3 - b*(x_1 + x_2 + x_3) - p_3

We substitute the solution of the second firm into the FOC of the third firm such that it only depends on x1 and x3. Afterwards, it is possible to write the amount the third firm produces only as a function of how much the leading firm produces. This can be substituted for x3 in the FOC of second firm such that it only depends on x1, too. 

In [299]:
foc3_twovariables = foc3_extend.subs(x_2, sol2_extend[0])
foc3_twovariables

a - b*x_3 - b*(x_1 + x_3 + (a - b*(x_1 + x_3) - p_2)/(2*b)) - p_3

Here, we solve the FOC of the third firm such that the solution only depends on how much the first firm produces:

In [300]:
foc3_twovariables
solution_foc_3 = sm.solve(sm.Eq(foc3_twovariables,0), x_3)
solution_foc_3

[(a - b*x_1 + p_2 - 2*p_3)/(3*b)]

Now, we substitute x3 by this in the FOC of the second firm:

In [301]:
solution_foc_2 = sol2_extend[0].subs(x_3, solution_foc_3[0])
solution_foc_2

(a - b*(x_1 + (a - b*x_1 + p_2 - 2*p_3)/(3*b)) - p_2)/(2*b)

Finally, we can substitute x2 and x3 in the objective function of the leading firm and solve the FOC for the leader directly: 

In [302]:
objective_1_only_onevariable = objective_1_extend.subs(x_2, solution_foc_2).subs(x_3, solution_foc_3[0])
objective_1_only_onevariable

-p_1*x_1 + x_1*(a - b*(x_1 + (a - b*(x_1 + (a - b*x_1 + p_2 - 2*p_3)/(3*b)) - p_2)/(2*b) + (a - b*x_1 + p_2 - 2*p_3)/(3*b)))

We derive the objective function and solve the FOC of the leader:

In [303]:
foc1_extend = sm.diff(objective_1_only_onevariable, x_1)
foc1_extend

a - b*x_1/3 - b*(x_1 + (a - b*(x_1 + (a - b*x_1 + p_2 - 2*p_3)/(3*b)) - p_2)/(2*b) + (a - b*x_1 + p_2 - 2*p_3)/(3*b)) - p_1

In [304]:
sol1_extend = sm.solve(sm.Eq(foc1_extend,0), x_1)
sol1_extend

[(a - 3*p_1 + p_2 + p_3)/(2*b)]

The last step is to find the amount both followers produce after having the optimal amount for the leader. We substitute x1 by the optimal x1: 

First the second firm:

In [305]:
optimal_second = solution_foc_2.subs(x_1, sol1_extend[0])

Here for the second firm:

In [306]:
optimal_third = solution_foc_3[0].subs(x_1, sol1_extend[0])

Now, we insert some values for the parameters a,b,c, $p_1$, $p_2$, $p_3$:
$$ a = 5 $$
$$ b = 1/4 $$ 
$$ p_1 = 2 $$
$$ p_2 = 1 $$
$$ p_3 = 1. $$

In [307]:
sol1_extend[0].subs(a, 5).subs(b, 1/4).subs(p_1, 2).subs(p_2,1).subs(p_3,1)

2.00000000000000

In [308]:
optimal_second.subs(a, 5).subs(b, 1/4).subs(p_1, 2).subs(p_2,1).subs(p_3,1)

4.66666666666667

In [309]:
optimal_third.subs(a, 5).subs(b, 1/4).subs(p_1, 2).subs(p_2,1).subs(p_3,1)

4.66666666666667

In [310]:
objective_1_extend.subs(a, 5).subs(b, 1/4).subs(p_1, 2).subs(p_2,1).subs(p_3,1)

x_1*(-0.25*x_1 - 0.25*x_2 - 0.25*x_3 + 5) - 2*x_1

### Numerical solution for the extension:

First, we turn the objective function of the leading firm into a Python function. This allows us to use a solver to find the optimal amount for the leading firm numerically.

We have to keep in mind that solvers usually minimize. Therefore, we define the negative objective function to minimize it (same as maximizing the objective function).

In [311]:
negative_objective = -objective_1_only_onevariable

In [312]:
negative_objective_1 = negative_objective.subs(a, 5).subs(b, 1/4).subs(p_1, 2).subs(p_2,1).subs(p_3,1)

In [313]:
sol_func = sm.lambdify(args=(x_1),expr= negative_objective_1)
sol_func

<function _lambdifygenerated(x_1)>

In [314]:
## find the solution numerically
def constraint_x(x):
    return x  # This ensures that x is non-negative


constraints ={'type': 'ineq', 'fun': constraint_x} ## adding constraint for x


## bounds for solutions
bounds = [(0,None)] ## non negative amount x

# c. call solver, use SLSQP
initial_guess = 10
## slsqp as a method can deal with bounds und constrains
sol_case2 = optimize.minimize(
sol_func, initial_guess, bounds = bounds ,constraints= constraints,
method='SLSQP')
sol_case2

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -0.3333333333333268
       x: [ 2.000e+00]
     nit: 4
     jac: [ 0.000e+00]
    nfev: 8
    njev: 4

In [315]:
np.round(sol_case2.x,2)[0] ## same solution as before

2.0

### Visualizing the extension

Here we visualize the model solution, again with an interactive plot.

In [316]:
## plotting profit of leader and followers
widgets.interact(model.interactive_figure_profit_extend, 
                 p1 =widgets.FloatSlider(description=r"p1", min=0, max=3, step=0.05, value=2),
                 p2 = widgets.FloatSlider(description=r"p2", min=0, max=3, step=0.05, value=1), 
                 p3 = widgets.FloatSlider(description=r"p3", min=0, max=3, step=0.05, value=1), 
                 a_1 =widgets.FloatSlider(description=r"a", min=2, max=10, step=0.05, value=5), 
                 b_1=widgets.FloatSlider(description=r"b", min=0, max=2, step=0.05, value=1/4));

interactive(children=(FloatSlider(value=2.0, description='p1', max=3.0, step=0.05), FloatSlider(value=1.0, des…

# Conclusion

Our starting point was a simple Stackelberg Duopol. We solved the model analytically and numerically. Then we visualized how the optimal amount changes depending on the chosen parameters for our cost function and inverse demand function.

In our setting, the follower produces more than the leader. That is due to the higher marginal costs of the leading firm. However, if both have the same cost function (as one can see in the interactive plots when changing the parameters) then the leader produces more. This makes sense since we have the sequential order of production in a Stackelberg Duopol. When the leader produces first and has the same costs of production, it would produce more than the follower.

We extended the model by adding a third firm to the market, having an oligopol now. Solving the model gets a bit more complicated than having only a duopol. We still have one leading firm and two followers in an oligopol with three firms. We assume that the two followers will produce at the same time. 
The solution is similiar in case of three firms in an oligopol to the one in a duopol.